# Семинар 2. Своя среда как MDP: считаем ценность и решаем уравнения Беллмана руками

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IlyaChichkanov/Reinforcement-learning/blob/main/02-environments/seminar/seminar.ipynb)

План:

1. Своя среда GridWorld как класс `gym.Env`: каркас, проверка, ручной маршрут
2. Модель MDP из среды: матрицы $P[s, a, s']$ и $R[s, a]$
3. Оценка политики тремя способами: Монте-Карло, линейная система, итерации Беллмана
4. $Q^\pi$ из $V^\pi$ и один шаг жадного улучшения
5. Уравнение оптимальности: value iteration на скользком поле; сравнение с Cross-Entropy
6. Q-learning на своём поле: то же уравнение, но по переходам
7. Что дальше: домашнее задание

Ячейки с `# TODO` заполняете сами; ниже каждой — проверка, которая должна пройти.

In [ ]:
# Если ноутбук открыт в Google Colab: ставим недостающие пакеты. Локально (после uv sync) ячейка ничего не делает.
import importlib.util, subprocess, sys
if importlib.util.find_spec("gymnasium") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium"], check=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env

GAMMA = 0.95

def draw_grid(env, values=None, policy=None, ax=None, title="", vmax=None):
    """Карта GridWorld; поверх — числа ценности и/или стрелки самого вероятного действия."""
    if ax is None:
        _, ax = plt.subplots(figsize=(3.4, 3.4 * env.n_rows / env.n_cols))
    colors = {".": "#ffffff", "#": "#555555", "S": "#f2f2f2", "G": "#a9dfa9"}
    grid = None if values is None else np.asarray(values, float).reshape(env.n_rows, env.n_cols)
    vmax = vmax or (1e-9 if grid is None else max(float(np.nanmax(grid)), 1e-9))
    for r in range(env.n_rows):
        for c in range(env.n_cols):
            ch = env.layout[r][c]
            face = colors[ch]
            if grid is not None and ch not in "#G" and not np.isnan(grid[r, c]):
                face = plt.cm.YlOrRd(0.85 * grid[r, c] / vmax)
            ax.add_patch(plt.Rectangle((c, r), 1, 1, facecolor=face, edgecolor="k", lw=0.6))
            if ch in "#G":
                continue                                        # у цели ценность 0 по определению, число не пишем
            if grid is not None and not np.isnan(grid[r, c]):
                ax.text(c + 0.5, r + 0.3, f"{grid[r, c]:.2f}", ha="center", va="center", fontsize=7.5)
            if policy is not None and ch != "G":
                ax.text(c + 0.5, r + 0.68, env.ARROWS[int(np.argmax(policy[r * env.n_cols + c]))],
                        ha="center", va="center", fontsize=14)
    ax.set_xlim(0, env.n_cols); ax.set_ylim(env.n_rows, 0); ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal")
    ax.set_title(title, fontsize=10)
    return ax

## 1. Своя среда GridWorld

На лекции среда была задана двумя массивами $P$ и $R$. В коде среда обычно задаётся иначе — классом с методами `reset` и `step`, и матриц в ней нигде нет. Сегодня мы пройдём путь в обе стороны: напишем среду как класс, а потом достанем из неё MDP.

Карта задаётся строками: `S` — старт, `G` — цель, `#` — стена, `.` — пол. Состояние — номер клетки `row * n_cols + col`, действия `0..3` = ←, ↓, →, ↑. С вероятностью `slip` действие заменяется случайным (равновероятно из четырёх). Награда: 1 при приходе в цель, иначе 0; эпизод заканчивается в цели.

Заполните `reset` и `step`. Подсказки:

* `super().reset(seed=seed)` создаёт `self.np_random` — используйте **его**, а не `np.random`, иначе среда не будет воспроизводимой;
* шаг в стену или за границу оставляет агента на месте;
* `step` возвращает ровно пять значений: `obs, reward, terminated, truncated, info`.

In [ ]:
class GridWorldEnv(gym.Env):
    metadata = {"render_modes": ["ansi", "rgb_array"], "render_fps": 4}
    MOVES = {0: (0, -1), 1: (1, 0), 2: (0, 1), 3: (-1, 0)}   # ←, ↓, →, ↑  как (dr, dc)
    ARROWS = "←↓→↑"
    DEFAULT_LAYOUT = ["S....#", ".##..#", "...#..", ".#..#.", ".#.#..", "....#G"]

    def __init__(self, layout=None, slip=0.0, render_mode=None):
        self.layout = [list(row) for row in (layout or self.DEFAULT_LAYOUT)]
        self.n_rows, self.n_cols = len(self.layout), len(self.layout[0])
        self.slip = slip
        self.render_mode = render_mode
        self.observation_space = spaces.Discrete(self.n_rows * self.n_cols)
        self.action_space = spaces.Discrete(4)
        self.start = self._find("S")
        self.goal = self._find("G")
        self.pos = self.start

    def _find(self, char):
        for r, row in enumerate(self.layout):
            for c, cell in enumerate(row):
                if cell == char:
                    return (r, c)
        raise ValueError(f"на карте нет клетки {char!r}")

    def _obs(self):
        return self.pos[0] * self.n_cols + self.pos[1]

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # TODO: вернуть агента на старт и вернуть (obs, info)
        raise NotImplementedError

    def step(self, action):
        # TODO: 1) с вероятностью self.slip заменить action случайным (через self.np_random)
        #       2) сдвинуть агента, если клетка свободна и внутри поля
        #       3) terminated = агент в цели; reward = 1.0 если terminated иначе 0.0
        #       4) вернуть (obs, reward, terminated, False, {})
        raise NotImplementedError

    def render(self):
        if self.render_mode == "ansi":
            return "\n".join("".join("A" if (r, c) == self.pos else ch for c, ch in enumerate(row))
                             for r, row in enumerate(self.layout))
        if self.render_mode == "rgb_array":
            return self._render_rgb()

    def _render_rgb(self):
        colors = {".": "#f4f4f4", "#": "#555555", "S": "#cfe3f7", "G": "#a9dfa9"}
        fig, ax = plt.subplots(figsize=(2.6, 2.6 * self.n_rows / self.n_cols))
        for r, row in enumerate(self.layout):
            for c, ch in enumerate(row):
                ax.add_patch(plt.Rectangle((c, r), 1, 1, color=colors[ch], ec="white"))
        ax.add_patch(plt.Circle((self.pos[1] + 0.5, self.pos[0] + 0.5), 0.3, color="#4C72B0"))
        ax.set_xlim(0, self.n_cols); ax.set_ylim(self.n_rows, 0); ax.set_aspect("equal"); ax.axis("off")
        fig.tight_layout(pad=0); fig.canvas.draw()
        img = np.asarray(fig.canvas.buffer_rgba())[:, :, :3].copy()
        plt.close(fig)
        return img

In [ ]:
# Проверка 1: базовое поведение.
env = GridWorldEnv()
obs, info = env.reset(seed=0)
assert obs == 0 and isinstance(info, dict)
obs, r, term, trunc, info = env.step(2)          # → из старта
assert obs == 1 and r == 0.0 and not term and not trunc
obs, *_ = env.step(3)                            # ↑ в стену поля: остаёмся
assert obs == 1
env.reset(seed=0)
obs, *_ = env.step(1); obs, *_ = env.step(1)     # ↓ ↓
assert obs == 12, obs
print("OK: reset/step ведут себя правильно")

### Проверка интерфейса, отрисовка, ручной маршрут

`check_env` из Gymnasium — обязательный первый тест любой среды. Он ловит неправильные типы, невоспроизводимый `seed`, лишние значения из `step`.

In [ ]:
check_env(GridWorldEnv())
print("check_env: ok")

env = GridWorldEnv(render_mode="ansi")
env.reset(seed=0)
print(env.render())

env = GridWorldEnv(render_mode="rgb_array")
env.reset(seed=0)
draw_grid(env, title="карта по умолчанию"); plt.show()

Случайный агент — базовая линия: любая политика, которую мы обучим, должна быть лучше него. Ручной агент — таблица «клетка → действие», написанная руками: убедитесь, что вы сами умеете решать задачу, прежде чем заставлять агента.

Заполните `manual_policy` так, чтобы агент доходил до цели на карте по умолчанию. Достаточно задать действия для клеток, через которые проходит маршрут.

In [ ]:
def run_episode(env, policy, seed=0, max_steps=100):
    """policy(obs) -> action. Возвращает (return, число шагов)."""
    obs, _ = env.reset(seed=seed)
    total = 0.0
    for t in range(max_steps):
        obs, r, terminated, truncated, _ = env.step(policy(obs))
        total += r
        if terminated or truncated:
            return total, t + 1
    return total, max_steps

rng = np.random.default_rng(0)
random_policy = lambda obs: int(rng.integers(4))

results = [run_episode(GridWorldEnv(), random_policy, seed=s) for s in range(200)]
print(f"случайный агент: доходит в {np.mean([g for g, _ in results]):.0%} эпизодов "
      f"(лимит 100 шагов), в среднем за {np.mean([t for _, t in results]):.0f} шагов")

# TODO: заполните маршрут (номер клетки -> действие 0..3). Карта:
#   S....#
#   .##..#
#   ...#..
#   .#..#.
#   .#.#..
#   ....#G
route = {
    0: 1,   # клетка 0 (старт): вниз
    # ...
}
manual_policy = lambda obs: route.get(int(obs), 0)

In [ ]:
# Проверка 2: ручной маршрут доходит до цели не более чем за 12 шагов.
G, steps = run_episode(GridWorldEnv(), manual_policy)
assert G == 1.0, "ручная политика не доходит до цели"
assert steps <= 12, f"маршрут слишком длинный: {steps} шагов (кратчайший — 10)"
print(f"OK: ручная политика доходит за {steps} шагов")

## 2. Модель MDP из среды

Класс выше — это MDP, только записанный кодом. Достанем из него матрицы явно: `mdp_matrices(env)` должна вернуть `P` формы `(n_states, n_actions, n_states)` и `R` формы `(n_states, n_actions)`, **не вызывая** `reset` и `step`, — только по карте и параметру `slip`.

Подсказки:

* вспомогательная функция «куда приведёт действие `a` из клетки `(r, c)`» уже нужна в `step`; переиспользуйте логику;
* при `slip > 0` действие `a` выполняется с вероятностью `1 - slip + slip / 4` (своё действие тоже может выпасть при заносе), каждое из трёх других — с вероятностью `slip / 4`;
* цель — терминальное состояние: `P[goal, a, goal] = 1`, `R[goal, a] = 0` при любом `a` (поглощающее состояние с нулевой наградой, как на лекции);
* `R[s, a]` — **ожидаемая** награда: сумма по исходам вероятности исхода на награду за него.

In [ ]:
def mdp_matrices(env):
    """P[s, a, s'] и R[s, a] для GridWorldEnv по карте и slip, без запуска среды."""
    n_s, n_a = env.observation_space.n, env.action_space.n
    P, R = np.zeros((n_s, n_a, n_s)), np.zeros((n_s, n_a))
    goal_s = env.goal[0] * env.n_cols + env.goal[1]

    def move(s, a):
        # TODO: номер клетки после попытки шага a из клетки s (стена или граница -> остаёмся)
        raise NotImplementedError

    for s in range(n_s):
        r, c = divmod(s, env.n_cols)
        if env.layout[r][c] == "#":
            P[s, :, s] = 1.0                                  # недостижимые клетки: формально поглощающие
            continue
        for a in range(n_a):
            # TODO: если s == goal_s — поглощающее состояние без награды;
            #       иначе для каждого фактического действия b с вероятностью
            #       (1 - slip + slip/4 если b == a иначе slip/4) добавить переход s -> move(s, b)
            #       и награду 1.0, если move(s, b) == goal_s
            raise NotImplementedError
    return P, R

# Проверка 3
env = GridWorldEnv(slip=0.2)
P, R = mdp_matrices(env)
assert P.shape == (36, 4, 36) and R.shape == (36, 4)
assert np.allclose(P.sum(axis=2), 1.0), "каждая строка P — распределение вероятностей"
goal_s = env.goal[0] * env.n_cols + env.goal[1]
assert np.allclose(P[goal_s, :, goal_s], 1.0) and np.allclose(R[goal_s], 0.0), "цель — поглощающее состояние без награды"
P0, R0 = mdp_matrices(GridWorldEnv(slip=0.0))
assert np.allclose(P0.max(axis=2), 1.0), "без скольжения переходы детерминированные"
assert P0[0, 2, 1] == 1.0 and P0[0, 3, 0] == 1.0, "из старта → ведёт в клетку 1, ↑ — в стену, остаёмся"
assert abs(R[29, 1] - (1 - 0.2 + 0.2 / 4)) < 1e-9, "из клетки 29 шаг ↓ ведёт в цель с вероятностью 1 - slip + slip/4"
print("OK: матрицы MDP построены верно")

<details>
<summary>Зачем это упражнение, если алгоритмы недель 3–10 обходятся без $P$ и $R$?</summary>

Чтобы один раз увидеть, что класс с `step` и «две таблицы» — одно и то же, и чтобы иметь **эталон**: зная $P$ и $R$, мы можем посчитать точную ценность любой политики и сверять с ней всё, что дальше будем оценивать по выборкам. Такой эталон — лучший отладчик для RL-кода.

</details>

## 3. Оценка политики тремя способами

Функции из лекции: сыграть эпизод политикой-таблицей и оценить ценность по Монте-Карло. Реализуйте два других способа — точное решение и итерации Беллмана — и убедитесь, что все три согласуются.

In [ ]:
def run_session(env, policy, rng, max_steps=100):
    """Один эпизод политикой-таблицей: состояния (включая последнее), действия, награды."""
    s, _ = env.reset(seed=int(rng.integers(1_000_000)))
    states, actions, rewards = [s], [], []
    for _ in range(max_steps):
        a = int(rng.choice(len(policy[s]), p=policy[s]))
        s, r, terminated, truncated, _ = env.step(a)
        states.append(s); actions.append(a); rewards.append(r)
        if terminated or truncated:
            break
    return states, actions, rewards

def discounted_return(rewards, gamma=GAMMA):
    return sum(gamma ** t * r for t, r in enumerate(rewards))

def estimate_values_mc(env, policy, n_episodes=3000, gamma=GAMMA, seed=0):
    """V(s) ≈ средний return, считая от первого попадания в s (first-visit Monte-Carlo)."""
    rng = np.random.default_rng(seed)
    n_s = env.observation_space.n
    total, count = np.zeros(n_s), np.zeros(n_s)
    for _ in range(n_episodes):
        states, actions, rewards = run_session(env, policy, rng)
        for t, s in enumerate(states[:-1]):
            if s not in states[:t]:
                total[s] += discounted_return(rewards[t:], gamma); count[s] += 1
    return np.divide(total, count, out=np.full(n_s, np.nan), where=count > 0)


def policy_evaluation_exact(P, R, policy, gamma=GAMMA):
    """V^π = (I - γ P_π)^{-1} r_π."""
    # TODO: P_pi[s, s'] = Σ_a policy[s, a] P[s, a, s'];  r_pi[s] = Σ_a policy[s, a] R[s, a];  решить линейную систему
    raise NotImplementedError

def policy_evaluation_iterative(P, R, policy, gamma=GAMMA, n_iter=200):
    """V ← r_π + γ P_π V, начиная с нуля. Возвращает V и список max|V_k - V_{k-1}|."""
    # TODO
    raise NotImplementedError

# Проверка 4: три способа согласуются.
env = GridWorldEnv(slip=0.2)
P, R = mdp_matrices(env)
uniform = np.ones((36, 4)) / 4
V_exact = policy_evaluation_exact(P, R, uniform)
V_iter, deltas = policy_evaluation_iterative(P, R, uniform)
V_mc = estimate_values_mc(env, uniform)
assert np.abs(V_exact - V_iter).max() < 1e-6, "итерации должны сойтись к точному решению"
assert abs(V_exact[0] - V_mc[0]) < 0.02, "в стартовой клетке (посещается в каждом эпизоде) Монте-Карло должен быть точен"
assert np.nanmax(np.abs(V_exact - V_mc)) < 0.12, "в остальных клетках допускаем шум: редкие клетки видны в немногих эпизодах"
assert deltas[-1] < deltas[0] * GAMMA ** 100, "ошибка должна убывать геометрически"
print("OK: точное решение, итерации и Монте-Карло согласуются")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, (V, name) in zip(axes, [(V_mc, "Монте-Карло, 3000 эпизодов"), (V_exact, "линейная система"), (V_iter, "итерации Беллмана")]):
    draw_grid(env, values=V, ax=ax, title=name, vmax=float(np.nanmax(V_exact)))
plt.show()

**Обсудите:** у каких клеток Монте-Карло ошибается сильнее и почему (посмотрите на клетки у цели: их ценность самая большая, а посещений — меньше всего)? Что будет с Монте-Карло-оценкой клеток, до которых равномерная политика почти не доходит? Сколько эпизодов понадобилось бы, чтобы ошибка в каждой клетке была меньше 0.01?

## 4. $Q^\pi$ и один шаг жадного улучшения

По $V^\pi$ ценность действия считается в одну строку: $Q^\pi(s, a) = r(s, a) + \gamma \sum_{s'} P(s' \mid s, a) V^\pi(s')$. А по $Q^\pi$ можно построить новую политику — в каждой клетке выбирать действие с наибольшей ценностью. На лекции мы заметили, что $V^\pi(s) \le \max_a Q^\pi(s, a)$; на неделе 3 докажем, что жадная политика **не хуже исходной во всех клетках**. Проверьте это численно.

In [ ]:
def q_from_v(P, R, V, gamma=GAMMA):
    # TODO: Q[s, a] = R[s, a] + γ Σ_s' P[s, a, s'] V[s']   (одна строка с матричным умножением)
    raise NotImplementedError

def greedy_policy(Q):
    # TODO: детерминированная политика-таблица: 1 у argmax_a Q[s, a], иначе 0
    raise NotImplementedError

# Проверка 5: жадная по Q^π политика не хуже π во всех клетках.
Q_uniform = q_from_v(P, R, V_exact)
pi_greedy = greedy_policy(Q_uniform)
assert Q_uniform.shape == (36, 4) and np.allclose(pi_greedy.sum(axis=1), 1.0) and set(np.unique(pi_greedy)) <= {0.0, 1.0}
assert np.allclose((uniform * Q_uniform).sum(axis=1), V_exact), "V^π = Σ_a π(a|s) Q^π(s, a)"
V_greedy = policy_evaluation_exact(P, R, pi_greedy)
assert np.all(V_greedy >= V_exact - 1e-9), "жадное улучшение не должно ухудшать ни одну клетку"
print(f"OK: ценность старта выросла с {V_exact[0]:.3f} до {V_greedy[0]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.6))
draw_grid(env, values=V_exact, policy=uniform, ax=axes[0], title="π: равномерная", vmax=float(V_greedy.max()))
draw_grid(env, values=V_greedy, policy=pi_greedy, ax=axes[1], title="жадная по Q^π", vmax=float(V_greedy.max()))
plt.show()

Один шаг улучшения — и стрелки уже почти ведут к цели, хотя $Q$ считалось для **равномерной** политики. Если повторить «оценить → улучшить» несколько раз, получится policy iteration (неделя 3).

## 5. Уравнение оптимальности: value iteration

Реализуйте value iteration из лекции: $V \leftarrow \max_a \big[ r(s, a) + \gamma \sum_{s'} P(s' \mid s, a) V(s') \big]$ до сходимости. Затем сравните оптимальную политику со скользкого поля с той, что находит Cross-Entropy из недели 1 (код дан), — по качеству и по тому, сколько эпизодов каждому методу понадобилось.

In [ ]:
def value_iteration(P, R, gamma=GAMMA, tol=1e-10, max_iter=10_000):
    """Возвращает V*, Q* и число итераций."""
    # TODO
    raise NotImplementedError

def cross_entropy_method(env, n_iter=25, n_sessions=200, q=0.7, laplace=0.5, mix=0.5, seed=0, max_steps=100):
    """Табличный Cross-Entropy из недели 1. Возвращает политику и число сыгранных эпизодов."""
    rng = np.random.default_rng(seed)
    n_states, n_actions = env.observation_space.n, env.action_space.n
    policy = np.ones((n_states, n_actions)) / n_actions
    for _ in range(n_iter):
        sessions = [run_session(env, policy, rng, max_steps) for _ in range(n_sessions)]
        returns = np.array([discounted_return(r) for *_, r in sessions])
        threshold = np.quantile(returns, q)
        counts = np.full((n_states, n_actions), laplace)
        for (states, actions, _), G in zip(sessions, returns):
            if G >= threshold and G > returns.min():
                for s, a in zip(states[:-1], actions):
                    counts[s, a] += 1
        new_policy = counts / counts.sum(axis=1, keepdims=True)
        policy = mix * new_policy + (1 - mix) * policy
    return policy, n_iter * n_sessions

def success_rate(env, policy, n_episodes=1000, seed=7):
    rng = np.random.default_rng(seed)
    return np.mean([sum(run_session(env, policy, rng)[2]) > 0 for _ in range(n_episodes)])

# Проверка 6
env = GridWorldEnv(slip=0.3)
P, R = mdp_matrices(env)
V_star, Q_star, n_iter = value_iteration(P, R)
pi_star = greedy_policy(Q_star)
assert np.allclose(V_star, Q_star.max(axis=1)), "V*(s) = max_a Q*(s, a)"
assert np.all(V_star >= policy_evaluation_exact(P, R, uniform) - 1e-9), "V* не меньше ценности равномерной политики"
assert np.all(V_star >= policy_evaluation_exact(P, R, pi_greedy) - 1e-9), "V* не меньше ценности жадной политики из части 4"
assert np.allclose(policy_evaluation_exact(P, R, pi_star), V_star, atol=1e-6), "ценность жадной по Q* политики равна V*"
print(f"OK: value iteration сошёлся за {n_iter} итераций; V*(старт) = {V_star[0]:.3f}")

pi_cem, n_episodes_cem = cross_entropy_method(env)
print(f"успех на скользком поле: π* {success_rate(env, pi_star):.1%} (0 сыгранных эпизодов), "
      f"Cross-Entropy {success_rate(env, pi_cem):.1%} ({n_episodes_cem} эпизодов)")
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.6))
draw_grid(env, values=V_star, policy=pi_star, ax=axes[0], title="π* и V* (value iteration)")
draw_grid(env, policy=pi_cem, ax=axes[1], title="Cross-Entropy, 5000 эпизодов")
plt.show()

**Обсудите:** value iteration не сыграл ни одного эпизода, а Cross-Entropy — тысячи. Что value iteration «знал», чего не знал Cross-Entropy? В каких задачах это знание недоступно, и что тогда остаётся (подсказка: раздел 7 лекции)?

## 6. Q-learning на своём поле

В разделе 7 лекции матожидание в уравнении оптимальности заменили одним переходом $(s, a, r, s')$ и получили правило обновления

$$
Q(s, a) \leftarrow Q(s, a) + \alpha\,\big[\, r + \gamma \max_{a'} Q(s', a') - Q(s, a) \big].
$$

Реализуйте табличный Q-learning с $\varepsilon$-жадным выбором действий (расписание $\varepsilon$ уже задано: сначала почти случайные действия, потом всё более жадные). Агент не должен пользоваться матрицами $P$ и $R$ — только `env.reset()` и `env.step()`. Проверка сравнит выученную таблицу с $Q^*$ из раздела 5.

In [ ]:
def q_learning(env, n_episodes=5000, alpha=0.1, eps_min=0.1, gamma=GAMMA, seed=0, max_steps=100):
    """Табличный Q-learning; ε убывает от 1 до eps_min за первую половину эпизодов. Возвращает таблицу Q[s, a]."""
    rng = np.random.default_rng(seed)
    n_states, n_actions = env.observation_space.n, env.action_space.n
    Q = np.zeros((n_states, n_actions))
    for episode in range(n_episodes):
        eps = max(eps_min, 1 - episode / (n_episodes / 2))          # сначала много исследуем, потом пользуемся знаниями
        s, _ = env.reset(seed=int(rng.integers(1_000_000)))
        for _ in range(max_steps):
            # TODO: 1) выбрать действие ε-жадно (с вероятностью eps — случайное, иначе argmax_a Q[s, a]);
            #       2) сделать шаг в среде;
            #       3) сдвинуть Q[s, a] к Беллмановскому таргету r + γ max_a' Q[s', a'];
            #       4) перейти в s'; если эпизод закончился (terminated или truncated) — прервать цикл.
            raise NotImplementedError
    return Q

# Проверка 7
Q_learned = q_learning(env)
pi_learned = greedy_policy(Q_learned)
free = [s for s in range(env.observation_space.n) if env.layout[s // env.n_cols][s % env.n_cols] in ".S"]
agree = np.mean([Q_learned[s].argmax() == Q_star[s].argmax() for s in free])
assert abs(Q_learned[0].max() - V_star[0]) < 0.1, "оценка старта далека от V*(старт): проверьте таргет и обновление"
assert np.abs(Q_learned - Q_star).max() < 0.3, "выученная таблица далека от Q*: проверьте таргет и обновление"
assert success_rate(env, pi_learned) >= 0.9, "жадная политика по выученной Q должна доходить почти всегда"
print(f"OK: наибольшее расхождение с Q* {np.abs(Q_learned - Q_star).max():.2f}, жадные действия совпадают с π* в {agree:.0%} свободных клеток, "
      f"успех {success_rate(env, pi_learned):.1%}")
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.6))
draw_grid(env, values=Q_learned.max(axis=1), policy=pi_learned, ax=axes[0], title="Q-learning: 5000 эпизодов, без P и R")
draw_grid(env, values=V_star, policy=pi_star, ax=axes[1], title="value iteration: P и R известны")
plt.show()

**Обсудите:** что произойдёт, если с самого начала действовать жадно (заменить расписание на $\varepsilon = 0$)? А если $\alpha = 1$ (жёстко заменять оценку таргетом)? Проверьте оба варианта и объясните результат через раздел 7 лекции.

## 7. Что дальше

* **Домашнее задание** (`../homework/homework.ipynb`): среда «управление запасами» как `gym.Env`, её точная модель $P$ и $R$ из распределения спроса, ценность ручной политики точно и по Монте-Карло, оптимальная политика заказов через value iteration; теория — вывод уравнений Беллмана, MDP на бумаге и лотерея Колобка.
* **Неделя 3**: те же уравнения без $P$ и $R$ — Monte-Carlo, TD-обучение, SARSA, Q-learning; и почему value iteration сходится (сжимающие отображения).